# 3.1 XGBoost: the three warning approaches, 30 and 60 minutes
**Purpose.** Fit and tune the XGBoost version of the flow forecast, the occupancy forecast and the direct classifier. The XGBoost pair, direct against occupancy, carries the primary comparison (paper Section 5.1).

**Method.** Every model is fitted on 2022. Each approach and horizon receives the same 30 random-search candidates (seed 1), and the candidate whose warning score has the highest AP on the 2023 usable calm origins is kept (`tune_xgboost`, a tie keeps the earlier trial). The forecasts are fitted with squared error but selected by the AP of their warning. No rebalancing is used. Warning thresholds are calibrated in notebook 3.4, and the flow ablation is in notebook 3.2.

**Outputs.** The kept settings go to `models/xgboost-selected.json`, every trial to `models/xgboost-trials.csv` and the 2022 and 2023 scores to `data/interim/scores-xgboost.parquet`. 2024 is not read. All numbers here are validation observations, because 2023 chooses the hyperparameters.

## 1. Setup

In [1]:
import time

import polars as pl

from madrid_congestion.config import (
    HORIZONS,
    INTERIM_DATA_DIR,
    MODELS_DIR,
    PANEL_YEAR,
    PROCESSED_DATA_DIR,
    VALIDATION_YEAR,
)
from madrid_congestion.evaluation import forecast_mae, warning_table
from madrid_congestion.features import origin_rows
from madrid_congestion.frozen import check_design, save_frozen
from madrid_congestion.labels import add_labels, references, studied_districts, year_grid
from madrid_congestion.models import (
    SEARCH_SPACE,
    SEED,
    TRIALS,
    at_edge,
    fit_model,
    fit_target,
    score_model,
    tune_xgboost,
    xgboost_candidates,
)

ROUTES = ["occupancy", "flow", "direct"]
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, VALIDATION_YEAR)}

## 2. Rows at each horizon
The 2022 districts, thresholds and flow references, applied unchanged to 2023. One row is one usable calm origin with the 34 inputs of paper Table 3.4. The checks require the code to match the design manifest of notebook 3.0, and the rows to match paper Table 3.3 (2022) and notebook 3.0 (2023).

In [2]:
districts = studied_districts(series[PANEL_YEAR])
check_design(districts)  # the code still builds the frozen design: label rules, inputs, search space, candidates
refs = references(year_grid(series[PANEL_YEAR], districts, PANEL_YEAR))
labels = {year: add_labels(year_grid(df, districts, year), refs) for year, df in series.items()}
rows = {(year, minutes): origin_rows(df, districts, minutes) for year, df in labels.items() for minutes in HORIZONS}

counts = {key: (df.height, int(df[f"target_{key[1]}"].sum())) for key, df in rows.items()}
assert counts[PANEL_YEAR, 30] == (475_434, 8_080) and counts[PANEL_YEAR, 60] == (453_544, 15_510)  # Table 3.3
assert counts[VALIDATION_YEAR, 30] == (482_391, 8_380)  # notebook 3.0
pl.DataFrame([
    {"year": year, "minutes": minutes, "usable_origins": n, "positive": positive, "base_rate": round(positive / n, 4)}
    for (year, minutes), (n, positive) in counts.items()
])

year,minutes,usable_origins,positive,base_rate
i64,i64,i64,i64,f64
2022,30,475434,8080,0.017
2022,60,453544,15510,0.0342
2023,30,482391,8380,0.0174
2023,60,461884,16275,0.0352


## 3. The candidates
30 draws with seed 1 from `models.SEARCH_SPACE`: tree depth 2–8, learning rate 0.02–0.3 (log-uniform), 100–800 trees, row subsample 0.6–1 and minimum child weight 1–20. The same list is used for every approach, both horizons and both flow-ablation variants.

In [3]:
candidates = xgboost_candidates()
assert len(candidates) == TRIALS and SEED == 1
print(pl.DataFrame([{"setting": name, "drawn": kind, "lowest": low, "highest": high} for name, (kind, low, high) in SEARCH_SPACE.items()]))
pl.DataFrame(candidates).with_row_index("trial").with_columns(pl.col("learning_rate", "subsample").round(3))

shape: (5, 4)
┌──────────────────┬─────────────┬────────┬─────────┐
│ setting          ┆ drawn       ┆ lowest ┆ highest │
│ ---              ┆ ---         ┆ ---    ┆ ---     │
│ str              ┆ str         ┆ f64    ┆ f64     │
╞══════════════════╪═════════════╪════════╪═════════╡
│ max_depth        ┆ integer     ┆ 2.0    ┆ 8.0     │
│ learning_rate    ┆ log-uniform ┆ 0.02   ┆ 0.3     │
│ n_estimators     ┆ integer     ┆ 100.0  ┆ 800.0   │
│ subsample        ┆ uniform     ┆ 0.6    ┆ 1.0     │
│ min_child_weight ┆ integer     ┆ 1.0    ┆ 20.0    │
└──────────────────┴─────────────┴────────┴─────────┘


trial,max_depth,learning_rate,n_estimators,subsample,min_child_weight
u32,i64,f64,i64,f64,i64
0,5,0.262,458,0.658,17
1,8,0.047,709,0.931,9
2,3,0.089,386,0.611,18
3,7,0.086,673,0.915,7
4,2,0.068,312,0.654,8
5,4,0.035,452,0.9,6
6,2,0.074,296,0.992,15
7,8,0.142,305,0.711,11
8,7,0.277,212,0.806,6


## 4. Tuning on 2023
For each horizon and approach, every candidate is fitted on 2022 and scored on 2023, and `tune_xgboost` keeps the one with the highest 2023 AP. The forecasts predict the largest relative occupancy or flow in *t*+1 … *t*+*H*, and the classifier predicts the target.

In [4]:
models, trials, selected = {}, {}, {}
for minutes in HORIZONS:
    for route in ROUTES:
        start = time.time()
        models[minutes, route], trials[minutes, route], record = tune_xgboost(
            route, minutes, rows[PANEL_YEAR, minutes], rows[VALIDATION_YEAR, minutes], districts, candidates
        )
        selected.setdefault(str(minutes), {})[route] = record  # JSON keys are text; ap_2023 chose the candidate
        print(f"{minutes} min, {route}: {TRIALS} candidates in {time.time() - start:.0f} s")

30 min, occupancy: 30 candidates in 367 s
30 min, flow: 30 candidates in 368 s
30 min, direct: 30 candidates in 356 s
60 min, occupancy: 30 candidates in 346 s
60 min, flow: 30 candidates in 344 s
60 min, direct: 30 candidates in 340 s


**Tuning record.** The kept candidate per horizon and approach, the spread of 2023 AP across the 30 candidates, and whether a kept integer setting lies at an edge of the search space (`at_edge`). An edge is reported, and the search space is not changed.

In [5]:
pl.DataFrame([
    {
        "minutes": minutes, "route": route, **record,
        "ap_lowest": trials[minutes, route]["ap"].min(), "ap_median": trials[minutes, route]["ap"].median(),
        "at_edge": ", ".join(at_edge(record)) or "none",
    }
    for minutes in HORIZONS for route, record in selected[str(minutes)].items()
]).with_columns(pl.col("learning_rate", "subsample").round(3), pl.col("ap_2023", "ap_lowest", "ap_median").round(4))

minutes,route,trial,max_depth,learning_rate,n_estimators,subsample,min_child_weight,ap_2023,ap_lowest,ap_median,at_edge
i64,str,i64,i64,f64,i64,f64,i64,f64,f64,f64,str
30,"""occupancy""",1,8,0.047,709,0.931,9,0.6961,0.5434,0.6797,"""max_depth"""
30,"""flow""",13,5,0.08,787,0.659,16,0.5331,0.4509,0.5255,"""none"""
30,"""direct""",17,5,0.042,744,0.858,1,0.7202,0.6312,0.7055,"""min_child_weight"""
60,"""occupancy""",1,8,0.047,709,0.931,9,0.7578,0.5888,0.7369,"""max_depth"""
60,"""flow""",1,8,0.047,709,0.931,9,0.6332,0.533,0.6182,"""max_depth"""
60,"""direct""",1,8,0.047,709,0.931,9,0.7822,0.6909,0.7735,"""max_depth"""


**Reproducibility.** Every kept model is refitted from its saved settings with `fit_model`, as the 2024 run does. The two fits must give identical 2023 scores.

In [6]:
for (minutes, route), model in models.items():
    params = {name: selected[str(minutes)][route][name] for name in SEARCH_SPACE}
    refit = fit_model("xgboost", route, minutes, rows[PANEL_YEAR, minutes], districts, params)
    validation = rows[VALIDATION_YEAR, minutes]
    assert (score_model(refit, route, validation, districts) == score_model(model, route, validation, districts)).all()
print("all", len(models), "kept models refit to identical 2023 scores")

all 6 kept models refit to identical 2023 scores


## 5. AP on 2022 and 2023
AP of each kept model, with the base rate. The 2023 AP chose the candidate, so it is optimistic. 2022 is the training year and is shown for comparison only.

In [7]:
scores = {key: {route: score_model(models[key[1], route], route, df, districts) for route in ROUTES} for key, df in rows.items()}
ap = []
for (year, minutes), df in rows.items():
    table = warning_table(df[f"target_{minutes}"], scores[year, minutes]).select("score", "ap")
    ap.extend({"minutes": minutes, "year": year, **row} for row in table.iter_rows(named=True))
print({key: round(float(df[f"target_{key[1]}"].to_numpy().mean()), 4) for key, df in rows.items()}, "base rates")
(
    pl.DataFrame(ap)
    .pivot("year", index=["minutes", "score"], values="ap")
    .rename({str(PANEL_YEAR): "ap_2022", str(VALIDATION_YEAR): "ap_2023"})
    .with_columns(pl.col("ap_2022", "ap_2023").round(4))
)

{(2022, 30): 0.017, (2022, 60): 0.0342, (2023, 30): 0.0174, (2023, 60): 0.0352} base rates


minutes,score,ap_2022,ap_2023
i64,str,f64,f64
30,"""occupancy""",0.7771,0.6961
30,"""flow""",0.5535,0.5331
30,"""direct""",0.8348,0.7202
60,"""occupancy""",0.831,0.7578
60,"""flow""",0.6771,0.6332
60,"""direct""",0.9059,0.7822


**Forecast accuracy.** MAE of the two XGBoost forecasts against the realised largest value in *t*+1 … *t*+*H*, next to the naive forecast, in relative and original units. The forecasts were chosen by warning AP and not by MAE, so their MAE need not be the lowest possible.

In [8]:
mae = []
for (year, minutes), df in rows.items():
    for variable, reference in (("occupancy", "threshold"), ("flow", "flow_reference")):
        mae.append({
            "minutes": minutes, "year": year, "forecast": variable,
            **forecast_mae(scores[year, minutes][variable], df[fit_target(variable, minutes)], df[f"{variable}_lag0"], df[reference]),
        })
pl.DataFrame(mae).sort("minutes", "forecast", "year").with_columns(pl.exclude("minutes", "year", "forecast").round(4))

minutes,year,forecast,mae_relative,naive_mae_relative,mae_original_units,naive_mae_original_units,mae_reduction
i64,i64,str,f64,f64,f64,f64,f64
30,2022,"""flow""",0.0192,0.0395,12.0978,24.6971,0.5142
30,2023,"""flow""",0.0196,0.0386,12.308,24.1034,0.4929
30,2022,"""occupancy""",0.0262,0.0508,0.3283,0.6265,0.4838
30,2023,"""occupancy""",0.0298,0.0511,0.3761,0.635,0.4167
60,2022,"""flow""",0.0199,0.0587,12.5231,36.295,0.661
60,2023,"""flow""",0.0222,0.0579,13.9322,35.8071,0.6171
60,2022,"""occupancy""",0.03,0.0751,0.3743,0.9202,0.6003
60,2023,"""occupancy""",0.0348,0.0758,0.4372,0.9343,0.5402


## 6. Frozen outputs
The kept settings with their 2023 AP (the 2024 run refits from them and checks its 2023 replay against this AP), every trial, and the scores of both years for notebook 3.4.

In [9]:
save_frozen("xgboost-selected.json", selected)
pl.concat([table.select(pl.lit(minutes).alias("minutes"), pl.lit(route).alias("route"), pl.all()) for (minutes, route), table in trials.items()]).write_csv(MODELS_DIR / "xgboost-trials.csv")
INTERIM_DATA_DIR.mkdir(exist_ok=True)
pl.concat([
    df.select(
        pl.lit(year).alias("year"), pl.lit(minutes).alias("minutes"), "distrito", "slot_start", target=pl.col(f"target_{minutes}"),
        **{f"xgboost_{route}": pl.Series(scores[year, minutes][route]) for route in ROUTES},
    )
    for (year, minutes), df in rows.items()
]).write_parquet(INTERIM_DATA_DIR / "scores-xgboost.parquet")
print(sorted(path.name for path in MODELS_DIR.iterdir()))

['calendar-2022.parquet', 'design.json', 'references-2022.parquet', 'thresholds-2023.json', 'xgboost-selected.json', 'xgboost-trials.csv']


**Result (2023 validation, not test results).** The 2023 AP of the kept models, with base rates of 1.74% (30 min) and 3.52% (60 min):

| Horizon | Direct | Occupancy forecast | Flow forecast |
|---|---|---|---|
| 30 min | 0.720 | 0.696 | 0.533 |
| 60 min | 0.782 | 0.758 | 0.633 |

At 60 minutes, all three approaches keep the same candidate (trial 1: depth 8, learning rate 0.047, 709 trees), at the deepest trees of the search space. At 30 minutes, the occupancy forecast keeps the same candidate, and the direct classifier keeps the smallest minimum child weight (1). These edges are recorded, and the search space is unchanged. The 2022 AP is higher than the 2023 AP for every model, by 0.02 to 0.12, so 2022 is not a measure of skill. The XGBoost forecasts reduce the naive MAE by 42% (occupancy) and 49% (flow) at 30 minutes and by 54% and 62% at 60 minutes. Every kept model refits from its saved settings to identical 2023 scores.

**Conclusion.** The six XGBoost models are fixed. Which approach warns better is decided once, on 2024 (notebook 4.0).